In [1]:
import math
import seaborn as sns
import matplotlib.pyplot as plt
import geopandas as gpd
from sklearn.cluster import KMeans
import os
from pathlib import Path
import sys
import pandas as pd
import numpy as np


In [2]:
# Set pandas to see all columns
pd.set_option('display.max_columns', None)

# Set working directory to project root
#os.chdir(Path(r"c:/Users/sosettle/Move-Smart"))
os.chdir(Path(r"/datasets/_deepnote_work/movesmart"))


# Make sure src/ is on the import path so we can `import census_data_loader`
src_path = Path.cwd() / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
# Optional: verify
print("CWD:", Path.cwd())
print("In sys.path:", src_path in map(Path, sys.path))

np.set_printoptions(threshold=sys.maxsize)


CWD: /datasets/_deepnote_work/movesmart
In sys.path: True


In [3]:
# import final dataset
all_df = pd.read_csv('data/clustering_output/Dataset_with_Cluster_Labels.csv')
all_df.head()

,cbsa_code,cbsa_name,city,state,cbsa_type,contains_imputed,MedianHouseholdIncome_B19013,TotalPopulation_B01003,MedianAge_B01002,TotalPovertyUniverse_B17001,BelowPoverty_B17001,IndustryTotalEmployed_C24050,IndustryAgForestryMining_C24050,IndustryConstruction_C24050,IndustryManufacturing_C24050,IndustryWholesaleTrade_C24050,IndustryRetailTrade_C24050,IndustryTransportUtilities_C24050,IndustryInformation_C24050,IndustryFinanceRealEstate_C24050,IndustryProfSciMgmtAdminWaste_C24050,IndustryEducationHealthCare_C24050,IndustryArtsRecAccommodationFood_C24050,IndustryOtherServices_C24050,IndustryPublicAdmin_C24050,TotalEmployed_C24010,ManagementBusiness_C24010,ScienceEngineering_C24010,ServiceOccupations_C24010,SalesOffice_C24010,Construction_C24010,ProductionTransportation_C24010,TotalPopulation25Plus_B15003,BachelorsDegree_B15003,MastersDegree_B15003,ProfessionalDegree_B15003,DoctorateDegree_B15003,TotalRace_B02001,WhiteAlone_B02001,BlackAlone_B02001,AmericanIndianAlaskaNative_B02001,AsianAlone_B02001,NativeHawaiianPacificIslander_B02001,SomeOtherRace_B02001,TwoOrMoreRaces_B02001,MedianGrossRent_B25064,MedianHomeValue_B25077,LaborForce_B23025,Unemployed_B23025,population_growth,job_growth,median_rent_growth,median_home_value_growth,industry_ag_forestry_mining_share,industry_construction_share,industry_manufacturing_share,industry_wholesale_share,industry_retail_share,industry_transport_utilities_share,industry_information_share,industry_finance_real_estate_share,industry_prof_sci_mgmt_admin_share,industry_education_health_share,industry_arts_rec_accommodation_food_share,industry_other_services_share,industry_public_admin_share,white_collar_share,blue_collar_share,service_job_share,sales_share,bachelors_or_higher_share,age_22_34_total,age_65_plus_total,age_22_34_share,age_65_plus_share,unemployment_rate,poverty_rate,diversity_index,land_area_m2,water_area_m2,land_area_sqmi,water_area_sqmi,centroid_lat,centroid_lon,population_density_per_sq_mile,nat_walk_index_mean,population_density_mean,activity_density_mean,intersection_density_mean,nat_walk_index_pop_weighted,n_tracts,total_population_weight,obesity_share,physical_inactivity_share,depression_share,current_asthma_share,diabetes_share,stroke_share,coronary_heart_disease_share,arthritis_share,any_disability_share,current_smoking_share,binge_drinking_share,violent_crime_count,property_crime_count,total_crime_count,violent_crime_per_population,property_crime_per_population,station_id,station_name,station_distance_km,avg_annual_temp,jan_avg_temp,jul_avg_temp,annual_precipitation,annual_snowfall,n_months_available,has_complete_year,temp_seasonality,snow_binary,cluster_k6,cluster_k20
0,10100,"Aberdeen, SD Micro Area",Aberdeen,SD,Micro,0,70761,42112,38.2,40877,4246,22300,1636,1287,2515,686,2657,1178,253,1242,1078,5270,2105,1188,1205,22300,3765,2142,280,451,143,159,28106,6809,1627,373,409,42112,36446,204,1762,1167,29,473,2031,796,208200,23046,746,-0.017544,-0.018702,0.186289,0.237077,7.336323,5.771300,11.278027,3.076233,11.914798,5.282511,1.134529,5.569507,4.834081,23.632287,9.439462,5.327354,5.403587,26.488789,1.354260,1.255605,2.022422,32.797267,3940,8516,9.356003,20.222264,3.237004,10.387259,0.245995,7353004125,111384253,2839.011,43.006,45.519435,-98.700700,14.833335,7.504762,3.527752,3.513267,66.674619,7.427158,10.0,32315.0,34.901711,24.237020,20.380343,9.771295,11.137416,3.335928,6.596871,27.861677,30.505292,14.208278,17.126502,75.0,395.0,470.0,0.001781,0.009380,USC00390022,"ABERDEEN 3 E, SD US",23.445079,43.144932,12.6,72.1,21.90,0.0,12,True,59.5,0,1,18
1,10140,"Aberdeen, WA Micro Area",Aberdeen,WA,Micro,0,63539,76397,44.7,73378,10358,29889,1738,1619,2925,791,3074,1738,126,1154,1852,6664,3402,1563,3243,29889,3661,1704,479,745,261,297,56001,6461,2683,526,198,76397,61800,1241,3168,956,311,2814,6107,1018,279500,32004,2115,0.035625,0.045362,0.229469,0.451194,5.814848,5.416708,9.786209,2.646459,10.284720,5.814848,0.421560,3.860952,6.196259,22.295828,11.382114,5.229349,10.850146,17.949747,1.

In [4]:
drop_cols = [
    # # label cols
    # 'cbsa_code', 'cbsa_name', 'city', 'state', 'cbsa_type',
    # census raw counts
'contains_imputed','TotalPopulation_B01003','TotalPovertyUniverse_B17001',
 'BelowPoverty_B17001', 'IndustryTotalEmployed_C24050', 'IndustryAgForestryMining_C24050',
  'IndustryConstruction_C24050', 'IndustryManufacturing_C24050', 'IndustryWholesaleTrade_C24050',
   'IndustryRetailTrade_C24050', 'IndustryTransportUtilities_C24050', 'IndustryInformation_C24050',
    'IndustryFinanceRealEstate_C24050', 'IndustryProfSciMgmtAdminWaste_C24050',
     'IndustryEducationHealthCare_C24050', 'IndustryArtsRecAccommodationFood_C24050', 
     'IndustryOtherServices_C24050', 'IndustryPublicAdmin_C24050', 'TotalEmployed_C24010', 
     'ManagementBusiness_C24010', 'ScienceEngineering_C24010', 'ServiceOccupations_C24010', 
     'SalesOffice_C24010', 'Construction_C24010', 'ProductionTransportation_C24010', 
     'TotalPopulation25Plus_B15003', 'BachelorsDegree_B15003', 'MastersDegree_B15003', 
     'ProfessionalDegree_B15003', 'DoctorateDegree_B15003', 'TotalRace_B02001', 'WhiteAlone_B02001', 
     'BlackAlone_B02001', 'AmericanIndianAlaskaNative_B02001', 'AsianAlone_B02001', 
     'NativeHawaiianPacificIslander_B02001', 'SomeOtherRace_B02001', 'TwoOrMoreRaces_B02001',
     'LaborForce_B23025', 'Unemployed_B23025',  'age_22_34_total', 'age_65_plus_total',  
      'land_area_m2', 'water_area_m2', 'land_area_sqmi', 'water_area_sqmi', 
      #'centroid_lat', 'centroid_lon',
      # walkability extra data
       'nat_walk_index_mean', 'population_density_mean',  
       # places descriptive or repetitive data
       'n_tracts', 'total_population_weight', 
       # crime raw counts
       'violent_crime_count', 'property_crime_count', 'total_crime_count',
       # weather dexcriptive data or no longer needed
        'station_id', 'station_name', 'station_distance_km', 'snow_binary','n_months_available', 'has_complete_year'
     
]

# for skewed data
log_transform_cols = [
    # walk data
    'activity_density_mean', 'intersection_density_mean',
    # census data
    'population_density_per_sq_mile',
    # weather data
    'annual_snowfall',
    # crime data
    'violent_crime_per_population','property_crime_per_population']


In [5]:
standardized_df = all_df.copy()

# drop columns
standardized_df = standardized_df.drop(drop_cols, axis=1)
standardized_df = standardized_df[standardized_df["state"] != "PR"]

standardized_df.head()

,cbsa_code,cbsa_name,city,state,cbsa_type,MedianHouseholdIncome_B19013,MedianAge_B01002,MedianGrossRent_B25064,MedianHomeValue_B25077,population_growth,job_growth,median_rent_growth,median_home_value_growth,industry_ag_forestry_mining_share,industry_construction_share,industry_manufacturing_share,industry_wholesale_share,industry_retail_share,industry_transport_utilities_share,industry_information_share,industry_finance_real_estate_share,industry_prof_sci_mgmt_admin_share,industry_education_health_share,industry_arts_rec_accommodation_food_share,industry_other_services_share,industry_public_admin_share,white_collar_share,blue_collar_share,service_job_share,sales_share,bachelors_or_higher_share,age_22_34_share,age_65_plus_share,unemployment_rate,poverty_rate,diversity_index,centroid_lat,centroid_lon,population_density_per_sq_mile,activity_density_mean,intersection_density_mean,nat_walk_index_pop_weighted,obesity_share,physical_inactivity_share,depression_share,current_asthma_share,diabetes_share,stroke_share,coronary_heart_disease_share,arthritis_share,any_disability_share,current_smoking_share,binge_drinking_share,violent_crime_per_population,property_crime_per_population,avg_annual_temp,jan_avg_temp,jul_avg_temp,annual_precipitation,annual_snowfall,temp_seasonality,cluster_k6,cluster_k20
0,10100,"Aberdeen, SD Micro Area",Aberdeen,SD,Micro,70761,38.2,796,208200,-0.017544,-0.018702,0.186289,0.237077,7.336323,5.771300,11.278027,3.076233,11.914798,5.282511,1.134529,5.569507,4.834081,23.632287,9.439462,5.327354,5.403587,26.488789,1.354260,1.255605,2.022422,32.797267,9.356003,20.222264,3.237004,10.387259,0.245995,45.519435,-98.700700,14.833335,3.513267,66.674619,7.427158,34.901711,24.237020,20.380343,9.771295,11.137416,3.335928,6.596871,27.861677,30.505292,14.208278,17.126502,0.001781,0.009380,43.144932,12.6,72.1,21.90,0.0,59.5,1,18
1,10140,"Aberdeen, WA Micro Area",Aberdeen,WA,Micro,63539,44.7,1018,279500,0.035625,0.045362,0.229469,0.451194,5.814848,5.416708,9.786209,2.646459,10.284720,5.814848,0.421560,3.860952,6.196259,22.295828,11.382114,5.229349,10.850146,17.949747,1.866908,1.602596,2.492556,17.621114,7.186146,26.436902,6.608549,14.115948,0.335725,47.113732,-123.826735,40.178010,2.051642,55.295277,9.796710,37.025057,23.624243,27.132959,11.732004,13.412424,4.144563,7.906169,31.228663,35.458485,15.352285,15.579366,0.001165,0.009241,51.153425,42.3,60.9,85.49,0.0,18.6,3,7
2,10180,"Abilene, TX Metro Area",Abilene,TX,Metro,66464,34.4,1095,172800,0.040209,0.051462,0.205947,0.383507,2.628475,7.190354,6.251882,1.833032,13.000602,6.013500,1.095303,7.215447,7.803874,28.268343,7.919301,5.141524,5.638362,20.608752,1.955987,1.506825,2.299759,26.241749,12.639416,16.688921,2.727639,13.155802,0.467582,32.452022,-99.718743,64.969233,2.990712,71.435586,7.029975,35.394659,27.259091,23.362871,10.152727,12.225254,3.430680,6.325994,24.326645,32.558226,14.047483,17.682745,0.000006,0.000067,66.364110,46.1,84.9,26.05,3.6,38.8,5,6
3,10220,"Ada, OK Micro Area",Ada,OK,Micro,62564,37.5,880,170500,-0.005914,0.034174,0.205479,0.265776,3.213309,7.406563,8.278259,2.324521,9.799453,3.361440,1.031222,4.290109,9.206928,29.290109,5.657475,4.597767,11.542844,24.077028,2.865770,1.521194,3.646308,29.732408,8.742597,19.041879,3.518030,12.556478,0.527200,34.721421,-96.691804,52.971252,1.682061,53.649489,6.620740,39.905537,29.864919,25.613345,12.101740,12.410335,3.843983,7.454902,28.357352,37.775495,16.767846,14.412803,0.000079,0.000446,61.495616,40.6,81.3,42.84,0.0,40.7,5,9
4,10300,"Adrian, MI Micro Area",Adrian,MI,Micro,67013,42.2,965,181100,0.005218,-0.016591,0.212312,0.266434,2.953481,6.498113,20.278750,1.782547,11.629758,4.290392,1.277795,3.289982,7.234778,24.232641,8.205630,4.281297,4.044837,21.408758,2.964849,1.441499,3.378655,22.273211,8.636653,22.845896,5.400813,11.614173,0.201837,41.896022,-84.074356,131.827629,1.651286,36.105463,6.026868,40.794930,26.490344,27.941694,11.535894,12.395936,3.773098,7.485598,29.764913,33.424492,15.840312,16.260148,0.001407,0.0055

In [6]:
standardized_df.columns

Index(['cbsa_code', 'cbsa_name', 'city', 'state', 'cbsa_type',
       'MedianHouseholdIncome_B19013', 'MedianAge_B01002',
       'MedianGrossRent_B25064', 'MedianHomeValue_B25077', 'population_growth',
       'job_growth', 'median_rent_growth', 'median_home_value_growth',
       'industry_ag_forestry_mining_share', 'industry_construction_share',
       'industry_manufacturing_share', 'industry_wholesale_share',
       'industry_retail_share', 'industry_transport_utilities_share',
       'industry_information_share', 'industry_finance_real_estate_share',
       'industry_prof_sci_mgmt_admin_share', 'industry_education_health_share',
       'industry_arts_rec_accommodation_food_share',
       'industry_other_services_share', 'industry_public_admin_share',
       'white_collar_share', 'blue_collar_share', 'service_job_share',
       'sales_share', 'bachelors_or_higher_share', 'age_22_34_share',
       'age_65_plus_share', 'unemployment_rate', 'poverty_rate',
       'diversity_index', 'cen

In [ ]:
## potential new scaling process?

In [7]:
# for weather, we are going to create a metric of how far the ave weather is from what 
# is considered a mild temperature
standardized_df["_temp_distance"] = (standardized_df["avg_annual_temp"] - 65).abs()


# need to cut out the high and low values for certain cities so that the scales are not skewed

inv_columns = ["MedianHomeValue_B25077","MedianGrossRent_B25064","unemployment_rate",
"violent_crime_per_population","property_crime_per_population","obesity_share",
"physical_inactivity_share","depression_share", "current_asthma_share",
"diabetes_share", "stroke_share", "coronary_heart_disease_share",
"arthritis_share","any_disability_share", "current_smoking_share",
"temp_seasonality", "annual_snowfall", "annual_precipitation","_temp_distance"
]
feature_scores = {}

for col in inv_columns:
    s = standardized_df[col].astype(float)

    lo = s.quantile(0.05)
    hi = s.quantile(0.95)
    s = s.clip(lower=lo, upper=hi)

    smin = s.min()
    smax = s.max()

    if smax == smin:
        scaled = pd.Series(50.0, index=s.index)
    else:
        scaled = 100 * (s - smin) / (smax - smin)

    scaled = 100 - scaled

    feature_scores[f"{col}_feature_score"] = scaled


no_inv_columns = ["MedianHouseholdIncome_B19013","job_growth","population_growth",
"bachelors_or_higher_share","nat_walk_index_pop_weighted","diversity_index",
"population_density_per_sq_mile", "intersection_density_mean","activity_density_mean",
"avg_annual_temp"
]


for col in no_inv_columns:
    s = standardized_df[col].astype(float)

    lo = s.quantile(0.05)
    hi = s.quantile(0.95)
    s = s.clip(lower=lo, upper=hi)

    smin = s.min()
    smax = s.max()

    if smax == smin:
        scaled = pd.Series(50.0, index=s.index)
    else:
        scaled = 100 * (s - smin) / (smax - smin)

    feature_scores[f"{col}_feature_score"] = scaled


feature_scores

{'MedianHomeValue_B25077_feature_score': 0       75.203812
 1       56.331392
 2       84.573849
 3       85.182636
 4       82.376919
           ...    
 930     94.685019
 931     26.871361
 932     78.512440
 933     85.288512
 934    100.000000
 Name: MedianHomeValue_B25077, Length: 925, dtype: float64,
 'MedianGrossRent_B25064_feature_score': 0       93.088757
 1       66.816568
 2       57.704142
 3       83.147929
 4       73.088757
           ...    
 930     95.100592
 931     32.733728
 932     70.958580
 933     91.313609
 934    100.000000
 Name: MedianGrossRent_B25064, Length: 925, dtype: float64,
 'unemployment_rate_feature_score': 0      8.679432e+01
 1      2.203315e+01
 2      9.657829e+01
 3      8.139634e+01
 4      4.523154e+01
            ...     
 930    4.278328e+01
 931    1.181274e+01
 932    1.421085e-14
 933    6.053205e+01
 934    1.421085e-14
 Name: unemployment_rate, Length: 925, dtype: float64,
 'violent_crime_per_population_feature_score': 0       68.678

In [8]:
feature_scores_df = pd.DataFrame(feature_scores, index=standardized_df.index)

feature_scores_df = pd.concat([standardized_df[['cbsa_code', 'cbsa_name', 'city', 'state', 'cbsa_type',
'MedianHomeValue_B25077','MedianGrossRent_B25064','MedianHouseholdIncome_B19013','centroid_lat', 'centroid_lon', 'cluster_k6']],
pd.DataFrame(feature_scores, index=standardized_df.index)],axis=1)

In [15]:
feature_scores_df[feature_scores_df['cbsa_code'] == 19100]

,cbsa_code,cbsa_name,city,state,cbsa_type,MedianHomeValue_B25077,MedianGrossRent_B25064,MedianHouseholdIncome_B19013,centroid_lat,centroid_lon,cluster_k6,MedianHomeValue_B25077_feature_score,MedianGrossRent_B25064_feature_score,unemployment_rate_feature_score,violent_crime_per_population_feature_score,property_crime_per_population_feature_score,obesity_share_feature_score,physical_inactivity_share_feature_score,depression_share_feature_score,current_asthma_share_feature_score,diabetes_share_feature_score,stroke_share_feature_score,coronary_heart_disease_share_feature_score,arthritis_share_feature_score,any_disability_share_feature_score,current_smoking_share_feature_score,temp_seasonality_feature_score,annual_snowfall_feature_score,annual_precipitation_feature_score,_temp_distance_feature_score,MedianHouseholdIncome_B19013_feature_score,job_growth_feature_score,population_growth_feature_score,bachelors_or_higher_share_feature_score,nat_walk_index_pop_weighted_feature_score,diversity_index_feature_score,population_density_per_sq_mile_feature_score,intersection_density_mean_feature_score,activity_density_mean_feature_score,avg_annual_temp_feature_score
210,19100,"Dallas-Fort Worth-Arlington, TX Metro Area",Dallas-Fort Worth-Arlington,TX,Metro,330300,1509,87155,32.849171,-96.970489,4,42.885124,8.710059,62.129821,46.617104,20.44807,57.612447,55.774608,73.157087,83.528266,72.719662,91.40897,100.0,100.0,85.724974,85.660818,52.580645,95.619048,47.41031,93.203406,85.133733,64.982506,62.000815,81.010981,79.018344,100.0,100.0,92.138244,99.844702,88.956795


In [13]:
feature_scores_df.head()

,cbsa_code,cbsa_name,city,state,cbsa_type,MedianHomeValue_B25077,MedianGrossRent_B25064,MedianHouseholdIncome_B19013,centroid_lat,centroid_lon,cluster_k6,MedianHomeValue_B25077_feature_score,MedianGrossRent_B25064_feature_score,unemployment_rate_feature_score,violent_crime_per_population_feature_score,property_crime_per_population_feature_score,obesity_share_feature_score,physical_inactivity_share_feature_score,depression_share_feature_score,current_asthma_share_feature_score,diabetes_share_feature_score,stroke_share_feature_score,coronary_heart_disease_share_feature_score,arthritis_share_feature_score,any_disability_share_feature_score,current_smoking_share_feature_score,temp_seasonality_feature_score,annual_snowfall_feature_score,annual_precipitation_feature_score,_temp_distance_feature_score,MedianHouseholdIncome_B19013_feature_score,job_growth_feature_score,population_growth_feature_score,bachelors_or_higher_share_feature_score,nat_walk_index_pop_weighted_feature_score,diversity_index_feature_score,population_density_per_sq_mile_feature_score,intersection_density_mean_feature_score,activity_density_mean_feature_score,avg_annual_temp_feature_score
0,10100,"Aberdeen, SD Micro Area",Aberdeen,SD,Micro,208200,796,70761,45.519435,-98.700700,1,75.203812,93.088757,86.794322,68.678251,60.911640,54.111709,70.505329,83.504314,87.848251,74.391341,75.207909,64.189208,51.712047,74.171419,63.819459,0.000000,100.000000,78.409045,0.000000,49.403257,29.939943,31.572156,61.480973,38.605714,24.290580,0.456345,59.406615,48.732904,0.000000
1,10140,"Aberdeen, WA Micro Area",Aberdeen,WA,Micro,279500,1018,63539,47.113732,-123.826735,3,56.331392,66.816568,22.033152,80.098065,61.524852,41.059065,74.179029,16.935523,18.723350,46.146091,40.555104,32.251014,27.296610,41.618525,51.803011,100.000000,100.000000,0.000000,36.570497,33.663017,58.031103,56.356697,9.387655,80.616074,41.628310,5.098654,44.973084,23.049780,28.125917
2,10180,"Abilene, TX Metro Area",Abilene,TX,Metro,172800,1095,66464,32.452022,-99.718743,5,84.573849,57.704142,96.578294,100.000000,100.000000,51.081456,52.387529,54.101895,74.400864,60.885340,71.147457,70.796831,77.346058,60.679324,65.508419,54.516129,93.142857,69.940002,97.748702,40.038010,60.705861,58.493734,38.978650,31.563952,67.105666,9.639588,65.445415,39.550763,85.461065
3,10220,"Ada, OK Micro Area",Ada,OK,Micro,170500,880,62564,34.721421,-96.691804,5,85.182636,83.147929,81.396336,100.000000,100.000000,23.352163,36.765168,31.916218,5.688295,58.587477,53.435972,43.258949,48.117698,26.390896,36.934204,48.387097,100.000000,35.676095,87.258938,31.538019,53.125161,36.993525,50.960607,24.308521,78.625090,7.441953,42.885570,16.555626,67.109769
4,10300,"Adrian, MI Micro Area",Adrian,MI,Micro,181100,965,67013,41.896022,-84.074356,1,82.376919,73.088757,45.231537,75.619333,77.748570,17.884881,56.996291,8.962836,25.637229,58.766246,56.473660,42.510155,37.910886,54.986133,46.676862,24.838710,100.000000,57.858863,29.575978,41.234547,30.865682,42.182687,25.356338,13.779631,15.758443,21.885840,20.632768,16.014857,22.746519


In [14]:
# Affordability
feature_scores_df["affordability_score"] = (
    feature_scores_df["MedianHomeValue_B25077_feature_score"] * (1/3)
    + feature_scores_df["MedianGrossRent_B25064_feature_score"] * (1/3)
    + feature_scores_df["MedianHouseholdIncome_B19013_feature_score"] * (1/3)
)

# Job Growth
feature_scores_df["job_growth_score"] = (
    feature_scores_df["job_growth_feature_score"] * 0.50
    + feature_scores_df["population_growth_feature_score"] * 0.20
    + feature_scores_df["unemployment_rate_feature_score"] * 0.30
)

# Safety
feature_scores_df["safety_score"] = (
    feature_scores_df["violent_crime_per_population_feature_score"] * 0.50
    + feature_scores_df["property_crime_per_population_feature_score"] * 0.50
)

# Education
feature_scores_df["education_score"] = (
    feature_scores_df["bachelors_or_higher_share_feature_score"] * 1.0
)

# Health
feature_scores_df["health_score"] = (
    feature_scores_df["obesity_share_feature_score"] * 0.10
    + feature_scores_df["physical_inactivity_share_feature_score"] * 0.10
    + feature_scores_df["depression_share_feature_score"] * 0.10
    + feature_scores_df["current_asthma_share_feature_score"] * 0.08
    + feature_scores_df["diabetes_share_feature_score"] * 0.12
    + feature_scores_df["stroke_share_feature_score"] * 0.10
    + feature_scores_df["coronary_heart_disease_share_feature_score"] * 0.12
    + feature_scores_df["arthritis_share_feature_score"] * 0.08
    + feature_scores_df["any_disability_share_feature_score"] * 0.10
    + feature_scores_df["current_smoking_share_feature_score"] * 0.10
)

# Walkability
feature_scores_df["walkability_score"] = (
    feature_scores_df["nat_walk_index_pop_weighted_feature_score"] * 1.0
)

# Diversity
feature_scores_df["diversity_score"] = (
    feature_scores_df["diversity_index_feature_score"] * 1.0
)

# Urban
feature_scores_df["urban_score"] = (
    feature_scores_df["population_density_per_sq_mile_feature_score"] * 0.50
    + feature_scores_df["intersection_density_mean_feature_score"] * 0.25
    + feature_scores_df["activity_density_mean_feature_score"] * 0.25
)

# Weather warmth
feature_scores_df["weather_warmth_score"] = (
    feature_scores_df["avg_annual_temp_feature_score"] * 1.0
)

# Weather mildness
feature_scores_df["weather_mildness_score"] = (
    feature_scores_df["_temp_distance_feature_score"] * 0.40
    + feature_scores_df["temp_seasonality_feature_score"] * 0.30
    + feature_scores_df["annual_snowfall_feature_score"] * 0.20
    + feature_scores_df["annual_precipitation_feature_score"] * 0.10
)

feature_scores_df.drop(columns=["_temp_distance_feature_score"], inplace=True)

feature_scores_df

,cbsa_code,cbsa_name,city,state,cbsa_type,MedianHomeValue_B25077,MedianGrossRent_B25064,MedianHouseholdIncome_B19013,centroid_lat,centroid_lon,cluster_k6,MedianHomeValue_B25077_feature_score,MedianGrossRent_B25064_feature_score,unemployment_rate_feature_score,violent_crime_per_population_feature_score,property_crime_per_population_feature_score,obesity_share_feature_score,physical_inactivity_share_feature_score,depression_share_feature_score,current_asthma_share_feature_score,diabetes_share_feature_score,stroke_share_feature_score,coronary_heart_disease_share_feature_score,arthritis_share_feature_score,any_disability_share_feature_score,current_smoking_share_feature_score,temp_seasonality_feature_score,annual_snowfall_feature_score,annual_precipitation_feature_score,MedianHouseholdIncome_B19013_feature_score,job_growth_feature_score,population_growth_feature_score,bachelors_or_higher_share_feature_score,nat_walk_index_pop_weighted_feature_score,diversity_index_feature_score,population_density_per_sq_mile_feature_score,intersection_density_mean_feature_score,activity_density_mean_feature_score,avg_annual_temp_feature_score,affordability_score,job_growth_score,safety_score,education_score,health_score,walkability_score,diversity_score,urban_score,weather_warmth_score,weather_mildness_score
0,10100,"Aberdeen, SD Micro Area",Aberdeen,SD,Micro,208200,796,70761,45.519435,-98.700700,1,75.203812,93.088757,8.679432e+01,68.678251,60.911640,54.111709,70.505329,83.504314,87.848251,74.391341,75.207909,64.189208,51.712047,74.171419,63.819459,0.000000,100.000000,78.409045,49.403257,29.939943,31.572156,61.480973,38.605714,24.290580,0.456345,59.406615,48.732904,0.000000,72.565275,47.322699,64.794945,61.480973,69.926504,38.605714,24.290580,27.263052,0.000000,27.840904
1,10140,"Aberdeen, WA Micro Area",Aberdeen,WA,Micro,279500,1018,63539,47.113732,-123.826735,3,56.331392,66.816568,2.203315e+01,80.098065,61.524852,41.059065,74.179029,16.935523,18.723350,46.146091,40.555104,32.251014,27.296610,41.618525,51.803011,100.000000,100.000000,0.000000,33.663017,58.031103,56.356697,9.387655,80.616074,41.628310,5.098654,44.973084,23.049780,28.125917,52.270326,46.896837,70.811458,9.387655,39.704275,80.616074,41.628310,19.555043,28.125917,64.628199
2,10180,"Abilene, TX Metro Area",Abilene,TX,Metro,172800,1095,66464,32.452022,-99.718743,5,84.573849,57.704142,9.657829e+01,100.000000,100.000000,51.081456,52.387529,54.101895,74.400864,60.885340,71.147457,70.796831,77.346058,60.679324,65.508419,54.516129,93.142857,69.940002,40.038010,60.705861,58.493734,38.978650,31.563952,67.105666,9.639588,65.445415,39.550763,85.461065,60.772000,71.025165,100.000000,38.978650,63.432222,31.563952,67.105666,31.068838,85.461065,81.076891
3,10220,"Ada, OK Micro Area",Ada,OK,Micro,170500,880,62564,34.721421,-96.691804,5,85.182636,83.147929,8.139634e+01,100.000000,100.000000,23.352163,36.765168,31.916218,5.688295,58.587477,53.435972,43.258949,48.117698,26.390896,36.934204,48.387097,100.000000,35.676095,31.538019,53.125161,36.993525,50.960607,24.308521,78.625090,7.441953,42.885570,16.555626,67.109769,66.622861,58.380186,100.000000,50.960607,37.405513,24.308521,78.625090,18.581276,67.109769,72.987314
4,10300,"Adrian, MI Micro Area",Adrian,MI,Micro,181100,965,67013,41.896022,-84.074356,1,82.376919,73.088757,4.523154e+01,75.619333,77.748570,17.884881,56.996291,8.962836,25.637229,58.766246,56.473660,42.510155,37.910886,54.986133,46.676862,24.838710,100.000000,57.858863,41.234547,30.865682,42.182687,25.356338,13.779631,15.758443,21.885840,20.632768,16.014857,22.746519,65.566741,37.438840,76.683951,25.356338,41.435083,13.779631,15.758443,20.104826,22.746519,45.067891
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
930,49660,"Youngstown-Warren, OH Metro Area",Youngstown-Warren,OH,Metro,134600,779,55357,41.195092,-80.765080,1,94.685019,95.10059

In [15]:
feature_scores_df.columns

Index(['cbsa_code', 'cbsa_name', 'city', 'state', 'cbsa_type',
       'MedianHomeValue_B25077', 'MedianGrossRent_B25064',
       'MedianHouseholdIncome_B19013', 'centroid_lat', 'centroid_lon',
       'cluster_k6', 'MedianHomeValue_B25077_feature_score',
       'MedianGrossRent_B25064_feature_score',
       'unemployment_rate_feature_score',
       'violent_crime_per_population_feature_score',
       'property_crime_per_population_feature_score',
       'obesity_share_feature_score',
       'physical_inactivity_share_feature_score',
       'depression_share_feature_score', 'current_asthma_share_feature_score',
       'diabetes_share_feature_score', 'stroke_share_feature_score',
       'coronary_heart_disease_share_feature_score',
       'arthritis_share_feature_score', 'any_disability_share_feature_score',
       'current_smoking_share_feature_score', 'temp_seasonality_feature_score',
       'annual_snowfall_feature_score', 'annual_precipitation_feature_score',
       'MedianHouseholdIn

In [16]:
standardized_indicies_df = feature_scores_df[['cbsa_code', 'cbsa_name', 'city', 'state', 'cluster_k6','centroid_lat', 'centroid_lon', 'MedianHomeValue_B25077',
    'MedianGrossRent_B25064','MedianHouseholdIncome_B19013','affordability_score', 'job_growth_score', 
    'safety_score','education_score', 'health_score', 'walkability_score','diversity_score', 'urban_score', 
    'weather_warmth_score','weather_mildness_score']]

In [17]:
standardized_indicies_df.head()

,cbsa_code,cbsa_name,city,state,cluster_k6,centroid_lat,centroid_lon,MedianHomeValue_B25077,MedianGrossRent_B25064,MedianHouseholdIncome_B19013,affordability_score,job_growth_score,safety_score,education_score,health_score,walkability_score,diversity_score,urban_score,weather_warmth_score,weather_mildness_score
0,10100,"Aberdeen, SD Micro Area",Aberdeen,SD,1,45.519435,-98.700700,208200,796,70761,72.565275,47.322699,64.794945,61.480973,69.926504,38.605714,24.290580,27.263052,0.000000,27.840904
1,10140,"Aberdeen, WA Micro Area",Aberdeen,WA,3,47.113732,-123.826735,279500,1018,63539,52.270326,46.896837,70.811458,9.387655,39.704275,80.616074,41.628310,19.555043,28.125917,64.628199
2,10180,"Abilene, TX Metro Area",Abilene,TX,5,32.452022,-99.718743,172800,1095,66464,60.772000,71.025165,100.000000,38.978650,63.432222,31.563952,67.105666,31.068838,85.461065,81.076891
3,10220,"Ada, OK Micro Area",Ada,OK,5,34.721421,-96.691804,170500,880,62564,66.622861,58.380186,100.000000,50.960607,37.405513,24.308521,78.625090,18.581276,67.109769,72.987314
4,10300,"Adrian, MI Micro Area",Adrian,MI,1,41.896022,-84.074356,181100,965,67013,65.566741,37.438840,76.683951,25.356338,41.435083,13.779631,15.758443,20.104826,22.746519,45.067891


In [18]:
!pwd

/datasets/_deepnote_work/movesmart


In [22]:
# standardized_indicies_df.to_csv("data/processed/standardized_indicies_df.csv")

In [14]:
def recommended_cities(
    df,
    user_inputs,
    user_income=None,
    housing_mode="either",
    top_n=10):

    ranked = df.copy()

    if user_income is not None:
        # 30% max is commonly known as the most you can spend on rent
        max_affordable_rent = (user_income/12)*0.30
        # 3-4x user_income is what your house value should be
        max_affordable_home_value = user_income*3.5
    
        if housing_mode in ["rent","either"]:
            ranked = ranked[ranked["MedianGrossRent_B25064"] <= max_affordable_rent]
        
        if housing_mode in ["buy","either"]:
            ranked = ranked[ranked["MedianHomeValue_B25077"] <= max_affordable_home_value]
    
    valid_inputs = {}
    for col, rating in user_inputs.items():
        if col in ranked.columns:
            if rating is not None:
                valid_inputs[col] = rating

    total = 0
    for rating in valid_inputs.values():
        total += rating

    weights = {}
    for col, rating in valid_inputs.items():
        weights[col] = (rating**2)/total

    ranked["recommendation_score"] = 0.0
    for col, weight in weights.items():
        ranked["recommendation_score"] += ranked[col] * weight

    ranked = ranked.sort_values("recommendation_score", ascending=False)

    return ranked.head(top_n)

In [15]:
#example user inputs for streamlit
user_inputs = {
    "affordability_score": 3,
    "safety_score": 1,
    "job_growth_score": 4,
    "education_score": 2,
    "health_score": 3,
    "walkability_score": 1,
    "diversity_score": 2,
    "urban_score": 1,
    "weather_warmth_score": 1,
    "weather_mildness_score": 2,
}

results = recommended_cities(
    df=standardized_indicies_df,
    user_inputs=user_inputs,
    user_income=50000,
    housing_mode="rent",
    top_n=10
)

results


,cbsa_code,cbsa_name,city,state,cluster_k6,centroid_lat,centroid_lon,MedianHomeValue_B25077,MedianGrossRent_B25064,MedianHouseholdIncome_B19013,affordability_score,job_growth_score,safety_score,education_score,health_score,walkability_score,diversity_score,urban_score,weather_warmth_score,weather_mildness_score,recommendation_score
641,37060,"Oxford, MS Micro Area",Oxford,MS,5,34.221062,-89.584602,222900,1036,61617,55.157767,99.608064,87.679954,77.478487,63.876290,9.888193,68.037269,13.208805,70.528046,72.791552,185.978488
391,26620,"Huntsville, AL Metro Area",Huntsville,AL,4,34.783262,-86.734871,265000,1091,83529,65.192612,81.452050,46.717885,94.334766,66.540632,32.550573,74.157928,47.846149,74.536002,75.335241,183.289718
54,12220,"Auburn-Opelika, AL Metro Area",Auburn-Opelika,AL,5,32.492206,-85.527127,224600,974,58991,55.545759,89.752274,53.767676,83.997977,61.140253,17.154052,78.171943,31.269397,76.687142,81.385445,181.965511
414,27600,"Jefferson, GA Micro Area",Jefferson,GA,3,34.130905,-83.562557,312700,1048,85012,63.757681,95.018712,99.127404,42.641083,70.775332,6.313317,44.684005,20.618258,66.235062,74.443386,178.523222
188,18100,"Columbus, NE Micro Area",Columbus,NE,0,41.574696,-97.353145,194600,877,75501,74.013525,100.000000,67.728358,23.831095,68.405969,48.759708,60.817964,33.598037,26.492166,32.051059,176.257709
355,25200,"Hailey, ID Micro Area",Hailey,ID,3,43.325486,-114.197340,544300,1166,81389,40.622866,97.274494,86.625970,84.730462,80.461652,82.293771,47.792808,11.065230,0.000000,36.844977,175.180526
794,43620,"Sioux Falls, SD-MN Metro Area",Sioux Falls,SD-MN,4,43.525658,-96.875416,274800,987,81418,66.896891,92.440608,18.613475,71.602006,79.155321,82.834605,31.599177,41.132218,12.546420,32.890807,174.650716
363,25580,"Hastings, NE Micro Area",Hastings,NE,1,40.404569,-98.349062,171800,832,67077,71.680325,100.000000,56.771460,42.529107,63.774127,41.235971,23.251507,28.341837,31.883956,48.656477,171.753583
285,22220,"Fayetteville-Springdale-Rogers, AR Metro Area",Fayetteville-Springdale-Rogers,AR,4,36.116394,-94.079545,273400,1094,77979,60.301089,78.243102,47.705049,68.928034,70.920868,31.762318,74.988418,33.440678,55.913103,63.341549,171.537020
251,20820,"Effingham, IL Micro Area",Effingham,IL,1,39.155051,-88.437879,167400,740,74219,80.886356,100.000000,82.700307,32.210234,63.198033,24.354957,0.000000,10.614624,41.815549,52.197991,169.693892


In [16]:
user_inputs = {
    "affordability_score": 3,
    "safety_score": 4,
    "job_growth_score": 2,
    "education_score": 2,
    "health_score": 3,
    "walkability_score": 1,
    "diversity_score": 3,
    "urban_score": 1,
    "weather_warmth_score": 3,
    "weather_mildness_score": 2,
}

results = recommended_cities(
    df=standardized_indicies_df,
    user_inputs=user_inputs,
    user_income=150000,
    housing_mode="rent",
    top_n=10
)

results

,cbsa_code,cbsa_name,city,state,cluster_k6,centroid_lat,centroid_lon,MedianHomeValue_B25077,MedianGrossRent_B25064,MedianHouseholdIncome_B19013,affordability_score,job_growth_score,safety_score,education_score,health_score,walkability_score,diversity_score,urban_score,weather_warmth_score,weather_mildness_score,recommendation_score
406,27260,"Jacksonville, FL Metro Area",Jacksonville,FL,4,30.234184,-81.756033,308900,1416,77013,43.764957,71.842552,100.000000,69.460847,68.648795,69.505099,83.520346,78.629056,100.000000,76.965902,220.192426
631,36740,"Orlando-Kissimmee-Sanford, FL Metro Area",Orlando-Kissimmee-Sanford,FL,4,28.434396,-81.356084,338500,1659,75611,33.562808,65.307807,89.590292,71.885342,79.734025,78.051654,100.000000,86.532632,100.000000,77.428149,219.841068
50,12060,"Atlanta-Sandy Springs-Roswell, GA Metro Area",Atlanta-Sandy Springs-Roswell,GA,4,33.732402,-84.392848,335100,1563,86338,42.429077,58.682448,89.681593,91.187339,83.030530,57.860356,100.000000,75.803445,77.398681,79.944090,217.231225
833,45220,"Tallahassee, FL Metro Area",Tallahassee,FL,5,30.346366,-84.267001,249900,1197,63078,47.485879,46.085813,94.193930,89.144144,71.086618,59.793544,88.398543,37.292587,95.195410,79.366682,215.919733
762,42340,"Savannah, GA Metro Area",Savannah,GA,4,32.109153,-81.273107,271100,1370,74632,47.184866,57.184276,92.938453,68.859966,73.671993,70.502139,89.717025,68.155402,92.379204,81.009952,215.852806
811,44260,"Starkville, MS Micro Area",Starkville,MS,5,33.514617,-89.077508,184200,857,46930,55.808734,53.658175,100.000000,87.933477,66.171981,10.589190,79.563911,10.420649,76.544628,76.693564,208.206416
834,45300,"Tampa-St. Petersburg-Clearwater, FL Metro Area",Tampa-St. Petersburg-Clearwater,FL,4,28.120541,-82.525188,306100,1497,71254,36.632850,59.243201,90.053384,65.665102,67.901382,91.871270,79.034125,98.131564,100.000000,73.647951,207.383217
657,37860,"Pensacola-Ferry Pass-Brent, FL Metro Area",Pensacola-Ferry Pass-Brent,FL,3,30.608557,-87.158547,260500,1267,73588,51.424761,58.649454,100.000000,50.899307,61.810938,58.647702,67.148504,46.680484,99.074277,76.599938,206.877054
545,33100,"Miami-Fort Lauderdale-West Palm Beach, FL Metr...",Miami-Fort Lauderdale-West Palm Beach,FL,4,26.101828,-80.478755,405600,1770,73481,26.095133,47.694414,81.797345,71.148946,72.053751,100.000000,100.000000,100.000000,100.000000,67.021968,205.648283
49,12020,"Athens-Clarke County, GA Metro Area",Athens-Clarke County,GA,5,33.943984,-83.213897,280900,1144,62897,46.709980,59.360952,97.906794,98.210050,77.163710,26.451769,74.933339,40.928203,64.335904,74.419622,205.422398


In [17]:
user_inputs = {
    "affordability_score": 5,
    "safety_score": 2,
    "job_growth_score": 2,
    "education_score": 2,
    "health_score": 3,
    "walkability_score": 4,
    "diversity_score": 3,
    "urban_score": 5,
    "weather_warmth_score": 1,
    "weather_mildness_score": 2,
}

results = recommended_cities(
    df=standardized_indicies_df,
    user_inputs=user_inputs,
    user_income=35000,
    housing_mode="rent",
    top_n=10
)

results

,cbsa_code,cbsa_name,city,state,cluster_k6,centroid_lat,centroid_lon,MedianHomeValue_B25077,MedianGrossRent_B25064,MedianHouseholdIncome_B19013,affordability_score,job_growth_score,safety_score,education_score,health_score,walkability_score,diversity_score,urban_score,weather_warmth_score,weather_mildness_score,recommendation_score
524,32260,"Marshalltown, IA Micro Area",Marshalltown,IA,1,42.041700,-92.981403,134100,860,72785,78.048897,60.332936,8.098137e+01,22.520836,55.895634,75.204110,58.522769,23.900205,16.105147,33.788780,192.702061
130,15580,"Butte-Silver Bow, MT Micro Area",Butte-Silver Bow,MT,0,45.896232,-112.660071,223500,810,57504,61.031943,45.659593,1.776965e+01,44.011083,55.500585,100.000000,9.561783,45.678596,0.000000,41.171566,187.854218
23,11020,"Altoona, PA Metro Area",Altoona,PA,1,40.498683,-78.309597,156700,854,60594,67.434883,46.330498,9.930615e+01,31.573240,60.078290,49.077718,1.572525,60.983575,28.584441,36.069237,187.319422
139,15940,"Canton-Massillon, OH Metro Area",Canton-Massillon,OH,1,40.711051,-81.261468,177400,873,65666,68.543795,52.096025,6.284520e+01,32.083468,35.913692,37.647461,25.702813,70.651493,32.365200,31.384168,185.613475
902,48220,"Weatherford, OK Micro Area",Weatherford,OK,5,35.449463,-98.991295,147700,778,59108,70.147410,90.590763,8.189471e+01,49.541559,35.874276,46.691865,56.787650,29.248581,63.083224,69.559105,182.598438
827,44980,"Sunbury, PA Micro Area",Sunbury,PA,1,40.885514,-76.712649,167100,821,59922,67.330860,93.404154,8.519405e+01,24.657204,43.340545,39.397765,9.645515,58.507349,26.806110,36.183771,180.612306
297,22700,"Fort Dodge, IA Micro Area",Fort Dodge,IA,1,42.433579,-94.175831,138500,762,68054,78.089515,58.678935,1.421085e-14,19.308699,55.840411,92.499156,22.796897,25.294162,11.113016,30.322602,179.885109
920,49100,"Winona, MN Micro Area",Winona,MN,0,43.981353,-91.777044,222900,866,70198,68.097935,44.998372,6.543561e+01,58.943166,71.899103,35.401865,10.022090,49.417228,12.720949,15.272788,172.169666
418,27780,"Johnstown, PA Metro Area",Johnstown,PA,1,40.510209,-78.710488,115900,733,56292,72.501006,45.162570,6.089603e+01,27.823609,60.078290,41.983056,7.241947,50.704130,12.415266,19.771024,171.888539
264,21380,"Emporia, KS Micro Area",Emporia,KS,5,38.382134,-96.359301,137800,796,58788,70.078350,48.653236,4.293205e+01,44.503377,58.428849,35.813638,52.492069,34.621780,39.156315,52.726777,171.835442


In [32]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

# =========================================================
# USER INPUTS
# =========================================================
user_inputs = {
    "affordability_score": 2,
    "job_growth_score": 10,
    "safety_score": 5,
    "education_score": 5,
    "health_score": 5,
    "walkability_score": 8,
    "diversity_score": 8,
    "urban_score": 8,
    "weather_warmth_score": 4,
    "weather_mildness_score": 5,
}

df = recommended_cities(
    df=standardized_indicies_df,
    user_inputs=user_inputs,
    user_income=1000000,
    housing_mode="rent",
    top_n=50
)

user_inputs_scaled = {k: v * 10 for k, v in user_inputs.items()}

# =========================================================
# SETTINGS
# =========================================================
RADAR_COLS = list(user_inputs_scaled.keys())

DISPLAY_LABELS = {
    "affordability_score": "Affordability",
    "job_growth_score": "Job Growth",
    "safety_score": "Safety",
    "education_score": "Education",
    "health_score": "Health",
    "walkability_score": "Walkability",
    "diversity_score": "Diversity",
    "urban_score": "Urban",
    "weather_warmth_score": "Warmth",
    "weather_mildness_score": "Mildness",
}

# =========================================================
# HELPERS
# =========================================================
def round_df_numeric(df_in, decimals=2):
    df_out = df_in.copy()
    numeric_cols = df_out.select_dtypes(include=[np.number]).columns
    df_out[numeric_cols] = df_out[numeric_cols].round(decimals)
    return df_out

def prepare_plot_df(df):
    df = df.copy()
    df["city_state"] = df["city"].astype(str) + ", " + df["state"].astype(str)

    if "cluster_label" not in df.columns:
        df["cluster_label"] = "Cluster " + df["cluster_k6"].astype(str)

    return round_df_numeric(df, 2)

def get_top_n(df, n=25):
    return df.sort_values("recommendation_score", ascending=False).head(n).copy()

# =========================================================
# 1. RADAR CHART
# =========================================================
def plot_radar(df):
    df = prepare_plot_df(df)
    row = get_top_n(df, 1).iloc[0]

    categories = [DISPLAY_LABELS[c] for c in RADAR_COLS]
    user_vals = [round(user_inputs_scaled[c], 2) for c in RADAR_COLS]
    city_vals = [round(row[c], 2) for c in RADAR_COLS]

    categories += [categories[0]]
    user_vals += [user_vals[0]]
    city_vals += [city_vals[0]]

    fig = go.Figure()

    fig.add_trace(go.Scatterpolar(
        r=city_vals,
        theta=categories,
        fill="toself",
        name=row["city_state"],
        hovertemplate="%{theta}: %{r:.2f}<extra>" + row["city_state"] + "</extra>"
    ))

    fig.add_trace(go.Scatterpolar(
        r=user_vals,
        theta=categories,
        fill="toself",
        name="User Preferences",
        opacity=0.4,
        hovertemplate="%{theta}: %{r:.2f}<extra>User Preferences</extra>"
    ))

    fig.update_layout(
        title=f"User Preferences vs Top Recommendation: {row['city_state']}",
        polar=dict(radialaxis=dict(range=[0, 100])),
        template="plotly_white"
    )

    return fig

# =========================================================
# 2. FEATURE IMPORTANCE / CONTRIBUTION CHART
# =========================================================
def plot_contributions(df):
    df = prepare_plot_df(df)
    row = get_top_n(df, 1).iloc[0]

    data = []
    for col in RADAR_COLS:
        gap = abs(row[col] - user_inputs_scaled[col])
        score = max(0, 100 - gap)
        contribution = score * user_inputs_scaled[col] / 100

        data.append({
            "feature": DISPLAY_LABELS[col],
            "contribution": round(contribution, 2),
            "gap": round(gap, 2),
            "user_preference": round(user_inputs_scaled[col], 2),
            "city_value": round(row[col], 2)
        })

    cdf = pd.DataFrame(data).sort_values("contribution", ascending=False)

    fig = px.bar(
        cdf,
        x="contribution",
        y="feature",
        orientation="h",
        title=f"Feature Importance for Top Recommendation: {row['city_state']}",
        hover_data={
            "contribution": ':.2f',
            "gap": ':.2f',
            "user_preference": ':.2f',
            "city_value": ':.2f'
        }
    )

    fig.update_layout(
        template="plotly_white",
        xaxis_title="Contribution",
        yaxis_title="Feature",
        yaxis=dict(categoryorder="array", categoryarray=cdf["feature"].tolist()[::-1])
    )

    return fig

# =========================================================
# 3. INTERACTIVE MAP
# =========================================================
def plot_map(df):
    df = prepare_plot_df(df)
    top = get_top_n(df, 5).copy()

    size_vals = np.clip(df["recommendation_score"] / 12, 4, 10)

    fig = px.scatter_geo(
        df,
        lat="centroid_lat",
        lon="centroid_lon",
        color="cluster_label",
        size=size_vals,
        size_max=10,
        hover_name="city_state",
        hover_data={
            "recommendation_score": ':.2f',
            "cluster_label": True,
            "centroid_lat": ':.2f',
            "centroid_lon": ':.2f',
            "city": False,
            "state": False,
            "city_state": False
        },
        scope="usa",
        title="Interactive U.S. Map of Recommendations"
    )

    star_sizes = [28 if i == 0 else 14 for i in range(len(top))]
    star_names = ["#1 Match"] + ["Top Picks"] * (len(top) - 1)

    fig.add_trace(go.Scattergeo(
        lat=top["centroid_lat"],
        lon=top["centroid_lon"],
        text=top["city_state"],
        mode="markers+text",
        textposition="top center",
        marker=dict(
            size=star_sizes,
            symbol="star",
            color="yellow",
            line=dict(width=1.5, color="black")
        ),
        name="Top Picks",
        hovertemplate="%{text}<extra>%{customdata}</extra>",
        customdata=star_names,
        showlegend=True
    ))

    fig.update_layout(template="plotly_white")

    return fig

# =========================================================
# 4. CLUSTER PROFILE
# =========================================================
def plot_cluster_profile(df, top_n=25):
    df = prepare_plot_df(df)
    top = get_top_n(df, top_n)

    cluster = top.iloc[0]["cluster_label"]
    cluster_df = top[top["cluster_label"] == cluster].copy()

    cluster_med = cluster_df[RADAR_COLS].median().round(2)

    compare = pd.DataFrame({
        "feature": [DISPLAY_LABELS[c] for c in RADAR_COLS],
        "Cluster Median": [round(cluster_med[c], 2) for c in RADAR_COLS],
        "User Preferences": [round(user_inputs_scaled[c], 2) for c in RADAR_COLS]
    })

    long = compare.melt(id_vars="feature", var_name="series", value_name="value")

    fig = px.bar(
        long,
        x="value",
        y="feature",
        color="series",
        barmode="group",
        orientation="h",
        title=f"Cluster Profile Comparison: {cluster} vs User Preferences",
        hover_data={"value": ':.2f'}
    )

    fig.update_layout(
        template="plotly_white",
        xaxis_title="Score",
        yaxis_title="Feature"
    )

    return fig

# =========================================================
# 5. TOP RECOMMENDATIONS TABLE
# =========================================================
def plot_table(df, top_n=10):
    df = prepare_plot_df(df)
    top = get_top_n(df, top_n).copy()

    table_df = top[[
        "city_state",
        "cluster_label",
        "recommendation_score",
        "affordability_score",
        "job_growth_score",
        "safety_score",
        "walkability_score",
        "diversity_score",
        "urban_score"
    ]].copy()

    table_df.columns = [
        "City, State",
        "Cluster",
        "Recommendation",
        "Affordability",
        "Job Growth",
        "Safety",
        "Walkability",
        "Diversity",
        "Urban"
    ]

    table_df = round_df_numeric(table_df, 2)

    fig = go.Figure(data=[go.Table(
        columnwidth=[180, 110, 100, 95, 95, 85, 95, 85, 75],
        header=dict(
            values=list(table_df.columns),
            align="left",
            fill_color="#E8EEF7",
            font=dict(size=12)
        ),
        cells=dict(
            values=[table_df[col] for col in table_df.columns],
            align="left",
            height=30,
            font=dict(size=11)
        )
    )])

    fig.update_layout(
        title=f"Top {top_n} Recommended Cities",
        template="plotly_white",
        height=420
    )

    return fig

# =========================================================
# 6. DROPDOWN FEATURE MAP
# =========================================================
def plot_dropdown_map(df):
    df = prepare_plot_df(df)

    features = RADAR_COLS + ["recommendation_score"]

    fig = go.Figure()

    for i, f in enumerate(features):
        size_vals = 10

        fig.add_trace(go.Scattergeo(
            lat=df["centroid_lat"],
            lon=df["centroid_lon"],
            text=(
                df["city_state"]
                + "<br>" + DISPLAY_LABELS.get(f, f.replace("_", " ").title())
                + ": " + df[f].round(2).astype(str)
            ),
            mode="markers",
            marker=dict(
                size=size_vals,
                color=df[f],
                colorscale="Viridis",
                showscale=(i == 0),
                colorbar=dict(
                    title=DISPLAY_LABELS.get(f, f.replace("_", " ").title())
                ) if i == 0 else None
            ),
            visible=(i == 0),
            name=DISPLAY_LABELS.get(f, f.replace("_", " ").title()),
            hovertemplate="%{text}<extra></extra>"
        ))

    buttons = []
    for i, f in enumerate(features):
        visible = [False] * len(features)
        visible[i] = True
        buttons.append(dict(
            label=DISPLAY_LABELS.get(f, f.replace("_", " ").title()),
            method="update",
            args=[
                {"visible": visible},
                {
                    "title": f"U.S. Feature Map: {DISPLAY_LABELS.get(f, f.replace('_', ' ').title())}"
                }
            ]
        ))

    fig.update_layout(
        title=f"U.S. Feature Map: {DISPLAY_LABELS.get(features[0], features[0].replace('_', ' ').title())}",
        updatemenus=[dict(buttons=buttons, x=1.05, y=1)],
        geo=dict(scope="usa"),
        template="plotly_white"
    )

    return fig

# =========================================================
# RUN EVERYTHING
# =========================================================
df = prepare_plot_df(df)

plot_radar(df).show()
plot_contributions(df).show()
plot_map(df).show()
plot_cluster_profile(df).show()
plot_table(df).show()
plot_dropdown_map(df).show()

In [19]:
def cluster_retreival(cluster_df, cbsa_code, top_n=5):
    metro_row = cluster_df[cluster_df["cbsa_code"] == cbsa_code]
    
    if metro_row.empty:
        raise ValueError("CBSA code not found in cluster dataframe.")
        
    cluster_id = metro_row.iloc[0]["cluster"]

    similar = cluster_df[cluster_df["cluster"] == cluster_id].copy()
    similar = similar[similar["cbsa_code"] != cbsa_code]

    return similar.head(top_n)


In [20]:
top_cbsa = results.iloc[0]["cbsa_code"]

similar = cluster_retreival(
    cluster_df=cluster_df,
    cbsa_code=top_cbsa,
    top_n=5
)

NameError: name 'cluster_df' is not defined

In [39]:
plot_df.head()

NameError: name 'plot_df' is not defined

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=eeaff733-bf4c-45b4-9fa5-aa4724c7f5cd' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>